In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
import time
import gc

print("--- STARTING KAGGLE BATCH EXECUTION (CUSTOM GRADIENT TAPE) ---")

# ==========================================
# PART 1: DATA PIPELINE (MEMORY-SAFE)
# ==========================================
print("\n[1/3] Initializing Memory-Safe Data Pipeline...")

DATASET_PATH = "/kaggle/input/datasets/yashaswi15/aerosense-training-data/Delhi_NCR_Master_DataCube_Scaled.npy"

master_data = np.load(DATASET_PATH, mmap_mode='r')
total_days = master_data.shape[0]
train_split = int(total_days * 0.8) 
test_split = total_days - train_split 

print(f"Total Days: {total_days} (Train: {train_split}, Test: {test_split})")

class SpatiotemporalGenerator(Sequence):
    def __init__(self, data_cube, start_idx, end_idx, batch_size=16):
        self.data_cube = data_cube
        self.start_idx = start_idx
        self.end_idx = end_idx - 8 
        self.batch_size = batch_size
        self.indices = np.arange(self.start_idx, self.end_idx)
        
    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        X_batch, Y_batch = [], []
        
        for i in batch_indices:
            window = self.data_cube[i : i+7]
            X_flat = np.transpose(window, (1, 2, 0, 3)).reshape((141, 231, 42))
            X_batch.append(X_flat)
            Y_batch.append(self.data_cube[i+7])
            
        X_batch = np.nan_to_num(np.array(X_batch), nan=0.0)
        Y_batch = np.nan_to_num(np.array(Y_batch), nan=0.0)
        return X_batch, Y_batch

# Batch size 16 to maximize Kaggle GPU throughput
train_gen = SpatiotemporalGenerator(master_data, 0, train_split, batch_size=16)
test_gen = SpatiotemporalGenerator(master_data, train_split, total_days, batch_size=16)


# ==========================================
# PART 2: PADDED U-NET ARCHITECTURE
# ==========================================
print("\n[2/3] Building Spatiotemporal U-Net Architecture...")

def build_unet(input_shape=(141, 231, 42)):
    inputs = layers.Input(shape=input_shape)
    
    # Pad to 144x240 for clean Max Pooling
    x = layers.ZeroPadding2D(padding=((0, 3), (0, 9)))(inputs)
    
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    b = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    b = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(b)
    
    u1 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(b)
    u1 = layers.concatenate([u1, c2])
    c3 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u1)
    
    u2 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c3)
    u2 = layers.concatenate([u2, c1])
    c4 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u2)
    
    outputs_padded = layers.Conv2D(6, (1, 1), activation='sigmoid')(c4)
    
    # Crop back to exact 141x231 spatial target
    outputs = layers.Cropping2D(cropping=((0, 3), (0, 9)))(outputs_padded)
    
    model = Model(inputs=[inputs], outputs=[outputs], name="Spatiotemporal_UNet")
    return model

unet_model = build_unet()


# ==========================================
# PART 3: ADVANCED CUSTOM TRAINING LOOP
# ==========================================
print("\n[3/3] Initiating Custom GradientTape Training Loop...")

MODEL_SAVE_PATH = "/kaggle/working/Delhi_NCR_UNet_Best.keras"

# Cosine Decay Learning Rate Scheduler
initial_learning_rate = 0.001
decay_steps = 50 * len(train_gen)
lr_schedule = CosineDecay(initial_learning_rate, decay_steps)

# Optimizer with Gradient Clipping
optimizer = Adam(learning_rate=lr_schedule, clipnorm=1.0)

# Custom Loss Function (Mean Absolute Error) with float16/float32 Patch
def calculate_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.abs(y_true - y_pred))

# Compiled Training Step
@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        predictions = unet_model(x_batch, training=True)
        loss = calculate_loss(y_batch, predictions)
    gradients = tape.gradient(loss, unet_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, unet_model.trainable_variables))
    return loss

# Compiled Validation Step
@tf.function
def val_step(x_batch, y_batch):
    predictions = unet_model(x_batch, training=False)
    val_loss = calculate_loss(y_batch, predictions)
    return val_loss

# Loop Parameters
EPOCHS = 50
best_val_loss = float('inf')
patience = 7
patience_counter = 0

print(f"\n--- Starting 50-Epoch Backpropagation ---")

for epoch in range(EPOCHS):
    start_time = time.time()
    
    epoch_loss_avg = tf.keras.metrics.Mean()
    epoch_val_loss_avg = tf.keras.metrics.Mean()
    
    # Training
    for step in range(len(train_gen)):
        x_batch, y_batch = train_gen[step]
        loss_val = train_step(x_batch, y_batch)
        epoch_loss_avg.update_state(loss_val)
        
    # Validation
    for step in range(len(test_gen)):
        x_val, y_val = test_gen[step]
        v_loss = val_step(x_val, y_val)
        epoch_val_loss_avg.update_state(v_loss)
        
    # Metrics
    train_loss = epoch_loss_avg.result().numpy()
    val_loss = epoch_val_loss_avg.result().numpy()
    
    # Safely extract learning rate (Handles both old TF and new Keras 3)
    if callable(optimizer.learning_rate):
        current_lr = optimizer.learning_rate(optimizer.iterations).numpy()
    else:
        current_lr = optimizer.learning_rate.numpy()
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Time: {time.time() - start_time:.1f}s | "
          f"LR: {current_lr:.5f} | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f}")
    
    # Save Model & Early Stopping
    if val_loss < best_val_loss:
        print(f"  -> Val Loss improved from {best_val_loss:.5f} to {val_loss:.5f}. Saving weights!")
        best_val_loss = val_loss
        unet_model.save(MODEL_SAVE_PATH)
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\n[!] Early Stopping Triggered at Epoch {epoch+1}. Model has converged.")
            break
            
    # Force VRAM Garbage Collection after every epoch
    gc.collect()
    tf.keras.backend.clear_session()

print("\n--- BATCH EXECUTION COMPLETE ---")
print(f"Your fully trained model is saved at: {MODEL_SAVE_PATH}")

2026-04-17 13:54:33.620047: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776434073.851766      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776434073.914443      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776434074.415475      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776434074.415520      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776434074.415523      22 computation_placer.cc:177] computation placer alr

--- STARTING KAGGLE BATCH EXECUTION (CUSTOM GRADIENT TAPE) ---

[1/3] Initializing Memory-Safe Data Pipeline...
Total Days: 2922 (Train: 2337, Test: 585)

[2/3] Building Spatiotemporal U-Net Architecture...


I0000 00:00:1776434102.620059      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



[3/3] Initiating Custom GradientTape Training Loop...

--- Starting 50-Epoch Backpropagation ---


I0000 00:00:1776434109.062760      66 cuda_dnn.cc:529] Loaded cuDNN version 91002


Epoch 01/50 | Time: 117.9s | LR: 0.00100 | Train Loss: 0.06893 | Val Loss: 0.04999
  -> Val Loss improved from inf to 0.04999. Saving weights!
Epoch 02/50 | Time: 99.7s | LR: 0.00100 | Train Loss: 0.04668 | Val Loss: 0.04881
  -> Val Loss improved from 0.04999 to 0.04881. Saving weights!
Epoch 03/50 | Time: 99.2s | LR: 0.00099 | Train Loss: 0.04431 | Val Loss: 0.04537
  -> Val Loss improved from 0.04881 to 0.04537. Saving weights!
Epoch 04/50 | Time: 99.4s | LR: 0.00098 | Train Loss: 0.04274 | Val Loss: 0.04436
  -> Val Loss improved from 0.04537 to 0.04436. Saving weights!
Epoch 05/50 | Time: 98.1s | LR: 0.00098 | Train Loss: 0.04101 | Val Loss: 0.04106
  -> Val Loss improved from 0.04436 to 0.04106. Saving weights!
Epoch 06/50 | Time: 97.0s | LR: 0.00096 | Train Loss: 0.03994 | Val Loss: 0.03939
  -> Val Loss improved from 0.04106 to 0.03939. Saving weights!
Epoch 07/50 | Time: 90.1s | LR: 0.00095 | Train Loss: 0.03914 | Val Loss: 0.03810
  -> Val Loss improved from 0.03939 to 0.0381